<a href="https://colab.research.google.com/github/LINWOO0099/Categorical-Encoding/blob/api/ReAct_style_reasoning_action_observation_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import math


# -----------------------------
# Tool 1: Math Tool
# -----------------------------
def math_tool(operation, a, b=None):
    if operation == "sqrt":
        return math.sqrt(a)

    elif operation == "power":
        return a ** b

    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed")
        return a / b

    else:
        raise ValueError("Unsupported math operation")


# -----------------------------
# Tool 2: Search Tool
# -----------------------------
KNOWLEDGE_BASE = {
    "refund policy": "Refunds are processed within 5-7 business days.",
    "cancellation policy": "Tickets can be cancelled up to 24 hours before the event."
}


def search_tool(query):
    return KNOWLEDGE_BASE.get(
        query.lower(),
        "No information found for this query."
    )


# -----------------------------
# Tool 3: Conversion Tool
# -----------------------------
def conversion_tool(value, from_unit, to_unit):

    conversions = {
        ("km", "miles"): lambda x: x * 0.621371,
        ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32
    }

    key = (from_unit.lower(), to_unit.lower())

    if key not in conversions:
        raise ValueError("Unsupported conversion")

    return round(conversions[key](value), 2)


# -----------------------------
# TOOL REGISTRY
# -----------------------------
TOOL_REGISTRY = {

    "math_tool": {
        "function": math_tool,
        "description": "Performs mathematical operations"
    },

    "search_tool": {
        "function": search_tool,
        "description": "Searches customer support knowledge base"
    },

    "conversion_tool": {
        "function": conversion_tool,
        "description": "Converts values between supported units"
    }
}



# -----------------------------
# Execute Tool Call
# -----------------------------
def execute_tool_call(call):

    try:
        tool_name = call.get("tool")
        args = call.get("args", {})

        tool = TOOL_REGISTRY.get(tool_name)

        if tool is None:
            return {
                "status": "error",
                "result": None,
                "error": f"Unknown tool: {tool_name}"
            }

        result = tool["function"](**args)

        return {
            "status": "ok",
            "result": result,
            "error": None
        }


    except Exception as e:

        return {
            "status": "error",
            "result": None,
            "error": str(e)
        }



# -----------------------------
# Episodic Memory
# -----------------------------
class EpisodicMemory:

    def __init__(self, window_size):
        self.window_size = window_size
        self.turns = []


    def add_turn(self, turn):
        self.turns.append(turn)


    def get_context(self):

        # list comprehension used here
        return [
            turn for turn in self.turns[-self.window_size:]
        ]



# -----------------------------
# ReAct Agent Loop Simulation
# -----------------------------
def run_agent_loop(query, tool_call_sequence, max_turns, memory):

    observations = []

    for i, call in enumerate(tool_call_sequence):

        if i >= max_turns:
            break


        result = execute_tool_call(call)

        observations.append(result)


        memory.add_turn(
            {
                "turn": i,
                "call": call,
                "observation": result
            }
        )


    return observations



# -----------------------------
# Save Memory JSON
# -----------------------------
def save_memory_to_file(memory, filepath):

    with open(filepath, "w") as file:
        json.dump(
            memory.get_context(),
            file,
            indent=4
        )



# -----------------------------
# Load Memory JSON
# -----------------------------
def load_memory_from_file(filepath):

    with open(filepath, "r") as file:
        return json.load(file)



# =====================================================
# SAMPLE INPUT
# =====================================================

tool_call_sequence = [

    {
        "tool": "math_tool",
        "args": {
            "operation": "sqrt",
            "a": 144
        }
    },

    {
        "tool": "search_tool",
        "args": {
            "query": "Cancellation Policy"
        }
    },

    {
        "tool": "math_tool",
        "args": {
            "operation": "divide",
            "a": 10,
            "b": 0
        }
    },

    {
        "tool": "conversion_tool",
        "args": {
            "value": 100,
            "from_unit": "km",
            "to_unit": "miles"
        }
    },


    {
        "tool": "weather_tool",
        "args": {
            "city": "Mumbai"
        }
    }

]


max_turns = 4
window_size = 3



# -----------------------------
# Run Agent
# -----------------------------

memory = EpisodicMemory(window_size)


results = run_agent_loop(
    "cancel my ticket",
    tool_call_sequence,
    max_turns,
    memory
)



print("Tool Execution Results:")
print(json.dumps(results, indent=4))


print("\nMemory Context:")
print(json.dumps(memory.get_context(), indent=4))



# -----------------------------
# Save and Reload Memory
# -----------------------------

save_memory_to_file(
    memory,
    "memory.json"
)


loaded_memory = load_memory_from_file(
    "memory.json"
)


print("\nReloaded Memory:")
print(json.dumps(loaded_memory, indent=4))

Tool Execution Results:
[
    {
        "status": "ok",
        "result": 12.0,
        "error": null
    },
    {
        "status": "ok",
        "result": "Tickets can be cancelled up to 24 hours before the event.",
        "error": null
    },
    {
        "status": "error",
        "result": null,
        "error": "Division by zero is not allowed"
    },
    {
        "status": "ok",
        "result": 62.14,
        "error": null
    }
]

Memory Context:
[
    {
        "turn": 1,
        "call": {
            "tool": "search_tool",
            "args": {
                "query": "Cancellation Policy"
            }
        },
        "observation": {
            "status": "ok",
            "result": "Tickets can be cancelled up to 24 hours before the event.",
            "error": null
        }
    },
    {
        "turn": 2,
        "call": {
            "tool": "math_tool",
            "args": {
                "operation": "divide",
                "a": 10,
                "b": 